In [262]:
import pandas as pd
from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler

train = pd.read_csv('train_data.csv')
test = pd.read_csv('test_data.csv')

In [261]:
# T1-4
task1 = train.shape[0]
task2 = train[train['Gender'] == 'male'].shape[0]
task3 = train['Duration'].mean()
task4 = train[train['Age'] >= 75].shape[0]

In [291]:
# T5
y_train = train['Calories']
x_train = train.drop(columns=['User_ID', 'Calories'])
t5_filter = test[test['Subtask'] == 5]
x5_test = t5_filter.drop(columns=['User_ID', 'Subtask'])

numCols = x_train.select_dtypes(exclude='object').columns

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [3, 5],
    'model__learning_rate': [0.05, 0.1]
}

preprocessor = ColumnTransformer(
    transformers=[
        ('oneHot', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Gender']),
        ('scale', StandardScaler(), numCols)
    ]
)

pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', XGBRegressor(random_state=42))
])

gs = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=5,
)

gs.fit(x_train, y_train)
model = gs.best_estimator_
t5predictions = model.predict(x5_test)

In [292]:
# T6
t6_filter = test[test['Subtask'] == 6]
x6_test = t6_filter.drop(columns=['User_ID', 'Subtask'])
t6predictions = model.predict(x6_test)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [293]:
# DataFrame
rows = []
ansList = [task1, task2, task3, task4]
for i,ans in enumerate(ansList, 1):
    rows.append({
        'subtaskID': i,
        'datapointID': 1,
        'answer': ans
    })
for id, pred in zip(t5_filter['User_ID'], t5predictions):
    rows.append({
        'subtaskID': 5,
        'datapointID': id,
        'answer': pred.astype(int)
    })
for id, pred in zip(t6_filter['User_ID'], t6predictions):
    rows.append({
        'subtaskID': 6,
        'datapointID': id,
        'answer': pred.astype(int)
    })
sub = pd.DataFrame(rows)
sub.to_csv('submission.csv', index=False)